# Wiktor Zoga 

In [8]:
import torch
import numpy as np

torch.manual_seed(2137)
np.random.seed(2137)

### Part A — Basic Operations


1. Create tensors:
   * zeros
   * ones
   * random normal
   * random uniform
2. Perform and demonstrate:
   * elementwise multiplication
   * matrix multiplication
   * broadcasting
   * reshaping
   * transpose
   * concatenation
3. Show examples of incorrect shape operations and explain why they fail.

In [2]:
# zeros 
zeros = torch.zeros(2, 2)

print(f"zeros:\t{zeros}\n")

# ones 
ones = torch.ones_like(zeros)

print(f"ones:\t{ones}\n")

# random normal
normal = torch.randn(2)

print(f"normal:\t{normal}\n")

# random uniform

uniform = torch.rand(1)

print(f"uniform:\t{uniform}\n")

zeros:	tensor([[0., 0.],
        [0., 0.]])

ones:	tensor([[1., 1.],
        [1., 1.]])

normal:	tensor([ 0.1058, -1.1060])

uniform:	tensor([0.4757])



In [3]:
a = zeros + normal
b = ones * uniform

print(f"a:\t{a}\n\nb:\t{b}\n")

# elementwise multiplication

c = a * b

print(f"c:\t{c}\n")

# matrix multiplication (@ operator)!!!

d = torch.matmul(a, b) @ normal

print(f"d:\t{d}\n")

# broadcasting

e = zeros + normal

print(f"e:\t{e}\n")

# reshaping

f = ones.reshape(4)

print(f"f:\t{f}\n")

# transposing

g = torch.tensor([[0, 1], [2, 3]]).transpose(0, 1)

print(f"g:\t{g}\n")


# concatenation

h0 = torch.cat([zeros, ones, zeros], dim=0)
h1 = torch.cat([zeros, ones, zeros], dim=1)

print(f"h0:\t{h0}\n")
print(f"h1:\t{h1}\n")

a:	tensor([[ 0.1058, -1.1060],
        [ 0.1058, -1.1060]])

b:	tensor([[0.4757, 0.4757],
        [0.4757, 0.4757]])

c:	tensor([[ 0.0503, -0.5261],
        [ 0.0503, -0.5261]])

d:	tensor([0.4759, 0.4759])

e:	tensor([[ 0.1058, -1.1060],
        [ 0.1058, -1.1060]])

f:	tensor([1., 1., 1., 1.])

g:	tensor([[0, 2],
        [1, 3]])

h0:	tensor([[0., 0.],
        [0., 0.],
        [1., 1.],
        [1., 1.],
        [0., 0.],
        [0., 0.]])

h1:	tensor([[0., 0., 1., 1., 0., 0.],
        [0., 0., 1., 1., 0., 0.]])



In [ ]:
# Incorrect shape operations (interpreter dobrze tłuamczy powody)

try:
    torch.ones(2) + torch.ones(3)
except Exception as ex:
    print(f"Error: {ex}\n")

try:
    torch.eye(2) @ torch.ones(3, 2)
except Exception as ex:
    print(f"Error: {ex}\n")

try:
    torch.tensor([1, 2, 3]).reshape(2)
except Exception as ex:
    print(f"Error: {ex}\n")

try:
    torch.tensor([1, 2, 3]).transpose(0, 1)
except Exception as ex:
    print(f"Error: {ex}\n")

try:    
    torch.ones(2, 3) + torch.ones(3, 2) 
except Exception as ex:
    print(f"Error: {ex}\n")

Error: The size of tensor a (2) must match the size of tensor b (3) at non-singleton dimension 0

Error: mat1 and mat2 shapes cannot be multiplied (2x2 and 3x2)

Error: shape '[2]' is invalid for input of size 3

Error: Dimension out of range (expected to be in range of [-1, 0], but got 1)

Error: The size of tensor a (3) must match the size of tensor b (2) at non-singleton dimension 1



## Part B — Manual Linear Algebra
Implement manually (without using torch.matmul for part of it):  
* Dot product.  
* Matrix-vector multiplication. 
* Batch matrix multiplication. 
* Then verify correctness using built-in PyTorch functions.  

(bez boradcastingu!)

In [5]:
# Dot product

def dot_product(a, b):
    if (a.shape != b.shape):
        raise ValueError("Vectors must have the same shape.")
    return torch.sum(a * b) 

a = torch.tensor([1, 2, 3])
b = torch.tensor([4, 5, 6])

print(f"Dot product of a and b:\t{dot_product(a, b)}\n")

print(f"torch.dot(a, b):\t{torch.dot(a, b)}\n")

Dot product of a and b:	32

torch.dot(a, b):	32



In [6]:
# Matrix-vector multiplication

def matrix_vector_mul(matrix, vector):
    if (matrix.shape[1] != vector.shape[0]):
        raise ValueError("Number of columns in the matrix must be equal to the number of elements in the vector.")
    results = torch.zeros(matrix.shape[1])
    for i in range(matrix.shape[0]):
        results[i] += dot_product(matrix[i], vector)
    return results

m = torch.tensor([[1, 2], [3, 4]])
v = torch.tensor([5, 6])

print(f"Matrix-vector multiplication result:\t{matrix_vector_mul(m, v)}\n")

print(f"torch.mv(m, v):\t{torch.mv(m, v)}\n")

Matrix-vector multiplication result:	tensor([17., 39.])

torch.mv(m, v):	tensor([17, 39])



In [7]:
# Batch matrix-matrix multiplication

# R^{batch x m x n} @ R^{batch x n x p} -> R^{batch x m x p}

def matrix_mul_matrix(a, b):
    m = a.shape[0]
    n = a.shape[1]
    p = b.shape[1]
    if (a.shape[1] != b.shape[0]):
        raise ValueError("Number of columns in the first matrix must be equal to the number of rows in the second matrix.")
    results = torch.zeros(m, p)
    for i in range(m):
        for j in range(p):
            results[i][j] += dot_product(a[i], b[:, j])
    return results

def batch_matrix_mul(batch_a, batch_b): 
    if (batch_a.shape[0] != batch_b.shape[0]):
        raise ValueError("Batch sizes of both matrices must be the same.")
    
    batch_size = batch_a.shape[0]
    m = batch_a.shape[1]
    n = batch_a.shape[2]
    p = batch_b.shape[2]

    results = torch.zeros(batch_size, m, p)
    for i in range(batch_size):
        results[i] = matrix_mul_matrix(batch_a[i], batch_b[i])
    return results

batch_a = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
batch_b = torch.tensor([[[9, 10], [11, 12]], [[13, 14], [15, 16]]])

print(f"Batch matrix-matrix multiplication result:\n{batch_matrix_mul(batch_a, batch_b)}\n")

print(f"torch.bmm(batch_a, batch_b):\n{torch.bmm(batch_a, batch_b)}\n")

Batch matrix-matrix multiplication result:
tensor([[[ 31.,  34.],
         [ 71.,  78.]],

        [[155., 166.],
         [211., 226.]]])

torch.bmm(batch_a, batch_b):
tensor([[[ 31,  34],
         [ 71,  78]],

        [[155, 166],
         [211, 226]]])



Difference between elementwise and matrix multiplication.

elementwise to mnożenie "wejście po wejściu" w odpowiadających sobie komórkach.

matrix multiplication to mnożenie stricte macierzowe ... no absotlutnie co innego, widać chociażby po przykładach wyżej


Broadcasting rules in PyTorch.

Tensor op liczba działa zawsze 

Tensor op tensor działa w zależności od tego jakie shape mają, torch dopisuję jedynki w miejsce brakujących wymiarów (idąc od lewej), wymiary mogą się nie zgadzać ale tylko pod warunkiem, że w conjamniej jednym jest jedynka.

Computational complexity of multiplying an n×mn \times mn×m matrix with an m×km \times km×k matrix.
(dziwne sformwatowane to pytanie, rozumiem, że chodzi o mnozenie macierzy mxn @ nxk = mxk)

Standardowy algorytm przechodzi przez k kolumn prawej macierzy (wektor n wymiarowy) i mnozy go skalarnie z każdym wierszem (wektor n wymiaorwy) aby uzyksać wynik mxk wymiarowy.

Czyli złożonośc to k * m * n (n to koszt pojdeyńczego mnożenia, k * m to liczba mnożeń)